# Retrieval-Augmented Generation(RAG) for Research Paper

# 1. Importing Libabries

In [ ]:
from langchain_core.documents import Document
import os

# 2. Loading Documents

### 2.1 Load all text documents 

In [ ]:
from langchain_community.document_loaders.text import TextLoader

docs = TextLoader("dataset/python.txt" , encoding="utf-8").load()

#this wad for text 

In [ ]:
docs

### 2.2 for pdf document 

In [ ]:
from langchain_community.document_loaders.pdf import PyMuPDFLoader

pdf_docs = PyMuPDFLoader("dataset/research2.pdf").load()

pdf_docs

In [ ]:
def load_all_pdf():
    folder_path = "dataset/"
    all_pdf_docs = []
    num_docs = 0

    for filename in os.listdir(folder_path):
        if filename.endswith(".pdf"):
            file_path = os.path.join(folder_path, filename)
            pdf_docs = PyPDFLoader(file_path).load()
            all_pdf_docs.extend(pdf_docs)
            num_docs += len(pdf_docs)

    print(f"Total PDF documents loaded: {num_docs}")
    return all_pdf_docs

In [ ]:
all_pdf_docs = load_all_pdf()

# 3. Chuck creating 

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents, chunk_size=500, chunk_overlap=50):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    
    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

In [ ]:
chunks = split_documents(all_pdf_docs)

len(chunks)

# 4. Embeddings

In [ ]:
from sentence_transformers import SentenceTransformer

class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)
        print(f"Loaded embedding model: {model_name}")

    def generate_embeddings(self, texts):
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings for {len(texts)} texts.")
        return embeddings

In [ ]:
emb_manager = EmbeddingManager()

# 5. Vector Database for storing values

In [ ]:
### 5.1 vector store

In [ ]:
import chromadb
import uuid

class VectorStoreManager:
    def __init__(self,presistence_dir="vector_store",collection_name="documents"):
        self.persistance_dir = presistence_dir
        self.collection_name = collection_name
        self.collection = None
        self.client = None


    def initialize_store(self):
        os.makedirs(self.persistance_dir, exist_ok=True)
        #client 
        self.client = chromadb.PersistentClient(path=self.persistance_dir)

        #collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "vector store for RAG injection research pipeline"}
            )
        
        print(f"Initialized vector store at {self.persistance_dir} with collection '{self.collection_name}'")
        
    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must match.")
        
        # id store , embeddings store ,document, metadata store
        ids = []
        all_metadatas = []
        documents_texts = []
        embeddings_list = []

        for i,(doc,emb) in enumerate(zip(documents, embeddings)):
            docs_ids = str(uuid.uuid4())
            ids.append(docs_ids)

            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            all_metadatas.append(metadata)

            documents_texts.append(doc.page_content)

            embeddings_list.append(emb)

            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=all_metadatas,
                documents=documents_texts
            )
        print(f"Added {len(documents)} documents to vector store with collection '{self.collection_name}'") 
        print(f"Total documents in collection '{self.collection_name}': {self.collection.count()}")


In [ ]:
vector_store_manager = VectorStoreManager()

vector_store_manager.initialize_store()


In [ ]:
text = [doc.page_content for doc in chunks]

embedding = emb_manager.generate_embeddings(text)

vector_store_manager.add_documents(chunks, embedding)
